# Prepare factoid evidence-plus-answer SFT data

**Run All defaults to DEV ONLY.** In the first setup cell, `ANNOTATION_SPLITS = ["dev"]` skips every training configuration, API call, preview and export cell. Select `["train"]` for training annotation, or `["train", "dev"]` for both. After changing the choice, rerun setup. Existing training caches/exports are preserved.


Prepare the **1,130-question training pool** with its **full original mixed contexts**. Keep an audit of **every original gold alias**, including aliases absent from the context. The current 1,352 per-alias SFT rows are an occurrence-filtered subset, not a semantic annotation. The inspected pool has **1,463 original gold aliases**, including **111 without an occurrence match**. It overlaps **200** previously accepted questions and **67** previously held questions; held questions remain excluded by default.

Workflow: freeze inputs → locate aliases → judge support and completeness → resolve/exclude issues → export matched evidence-plus-answer and answer-only datasets. This notebook prepares data; it does **not** train a model.

Use the `bioasq` kernel. No GPU is needed for preparation or an API-backed judge. Run the cells in order; only enabled splits execute. When train is enabled, its defaults configure a **full annotation run**: `SMOKE_TEST=False`, `ANNOTATION_LIMIT=None`, and `ALLOW_PARTIAL_EXPORT=False`. Completed judgments are reused from the same run and judge cache. Rerun Configuration and the following cells after changing settings; editing a cell does not update the running kernel. For a small smoke run, set `SMOKE_TEST=True` and `ALLOW_PARTIAL_EXPORT=True`; this uses a separate smoke directory.

**Annotation provenance:** automatic judgments are model-assisted labels, not independent expert validation. Existing user-accepted annotations are hints. Missing/invalid judgments never become supported labels. Previously flagged questions and aliases stay excluded until explicitly resolved in a human review file.

**New:** a standalone dev-only annotation section is appended below. Run All in the default dev-only mode, or run setup then the dev section.


In [ ]:
# Run All processes ONLY these splits. Change explicitly to ["train"] or ["train", "dev"].
ANNOTATION_SPLITS = ["dev"]
assert ANNOTATION_SPLITS and set(ANNOTATION_SPLITS) <= {"train", "dev"}
print("Enabled annotation splits:", ANNOTATION_SPLITS)

from pathlib import Path
from datetime import datetime, timezone
import os
import sys
import json
import importlib
from collections import Counter


def find_project_root(start):
    for parent in [start, *start.parents]:
        for candidate in [parent, parent / "Task-Structured Counterfactual Preference Mining"]:
            if (candidate / "cse_dpo").is_dir() and (candidate / "src").is_dir():
                return candidate.resolve()
    raise RuntimeError("Open this notebook inside the BioASQ_MRES workspace")

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from cse_dpo import evidence_sft_data as prep
prep = importlib.reload(prep)  # Pick up helper edits in an existing notebook kernel.
print("Project:", PROJECT_ROOT)


## 1. Configuration

`JUDGE_ENDPOINT` is the complete URL of an **OpenAI-compatible chat/completions endpoint** supporting JSON-object responses. The helper uses `max_completion_tokens` for OpenAI and `max_tokens` for other endpoints; it can also be a locally served endpoint. OpenAI defaults are configured below using `gpt-4.1-mini-2025-04-14`, a small model for checking the pipeline; passing a smoke run does not validate biomedical judgment quality. `JUDGE_MODEL` remains configurable. Use an instruction-following model suitable for evidence annotation. The existing answer-only SFT model has not been validated as an evidence judge.

The default `API_KEY_FILE` reads `open_ai_api.txt` from the project root only when making an API request. The file may contain a single raw key or `OPENAI_API_KEY=...`; contents are never printed or written into run artifacts. File credentials take precedence over the environment variable. Set `API_KEY_FILE=None` to use the environment variable instead. Do not paste tokens into cells. The existing Hugging Face token is not an API-judge credential. API annotation sends the full question, resources, gold aliases and historical review hints to the configured service. It runs only when you execute the annotation cell. No texts are silently truncated; context-limit/provider errors stop the run and retain completed judgments.

For a local server or a different provider, set `API_KEY_FILE=None` and supply that backend's endpoint/model and credentials if needed. The OpenAI key file is restricted to the official OpenAI endpoint.

For manual annotation or another model interface, set `ANNOTATION_BACKEND="manual"` and import the structured review records described below. The helper also accepts a custom Python judge callback returning `(decision, raw_metadata)`.

API reference: [Chat Completions](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create). Model reference: [GPT-4.1 mini](https://developers.openai.com/api/docs/models/gpt-4.1-mini).


In [ ]:
if "train" in globals().get("ANNOTATION_SPLITS", ["dev"]):
    SMOKE_TEST = False
    SMOKE_QUESTIONS = 1  # Fixed first unheld question(s), not an additional batch on every rerun.
    RUN_NAME = "supported_train_evidence_v1"  # Reuse to resume unchanged inputs; change for a different source pool.
    RUN_DIR = PROJECT_ROOT / "Artifacts/Factoid_SFT/evidence_annotation" / (RUN_NAME + ("_smoke" if SMOKE_TEST else ""))
    ANNOTATION_BACKEND = "api"  # "api" or "manual"
    JUDGE_ENDPOINT = os.environ.get("EVIDENCE_JUDGE_ENDPOINT", "https://api.openai.com/v1/chat/completions")
    JUDGE_MODEL = os.environ.get("EVIDENCE_JUDGE_MODEL", "gpt-4.1-mini-2025-04-14")
    API_KEY_FILE = PROJECT_ROOT / "open_ai_api.txt"  # Path only; the key is never displayed.
    API_KEY_ENV = "EVIDENCE_JUDGE_API_KEY"
    JUDGE_MAX_OUTPUT_TOKENS = 6000
    JUDGE_TIMEOUT_SECONDS = 120
    ANNOTATION_LIMIT = None  # New calls per execution; None processes all pending unheld questions.
    CONTINUE_ON_INVALID = True  # Save invalid responses for review and continue; never export them.
    MANUAL_REVIEWS = RUN_DIR / "manual_reviews.jsonl"

    # Automatic, structurally validated judgments may populate a provisional training export.
    # Set False for an export containing human-reviewed decisions only.
    ACCEPT_MODEL_JUDGMENTS = True
    # Default: one first eligible original alias per question; every alias stays in the audit.
    # True retains all eligible aliases (questions with more aliases then get more examples).
    EXPORT_ALL_SUPPORTED_ALIASES = False
    RESOURCE_ORDER_SEED = 3407  # Same reproducible permutation in both experimental arms.
    ALLOW_PARTIAL_EXPORT = False  # True only for a deliberately small pilot export.

    assert RUN_NAME and RUN_NAME not in {".", ".."} and not any(c in RUN_NAME for c in "/\\")
    assert isinstance(SMOKE_QUESTIONS, int) and SMOKE_QUESTIONS > 0
    assert ANNOTATION_BACKEND in {"api", "manual"}
    assert ANNOTATION_LIMIT is None or ANNOTATION_LIMIT > 0

    print("Annotation mode:", "SMOKE" if SMOKE_TEST else "FULL")
    print("Maximum new API calls:", SMOKE_QUESTIONS if SMOKE_TEST else (ANNOTATION_LIMIT if ANNOTATION_LIMIT is not None else "all pending unflagged questions"))
    print("Partial export allowed:", ALLOW_PARTIAL_EXPORT)
    print("Run directory:", RUN_DIR)
else:
    print("Skipping train section (disabled in ANNOTATION_SPLITS).")


## 2. Freeze the pool and locate aliases

Read all gold aliases from the original 1,440-question training source, restricted to the 1,130 IDs in the supported SFT pool. This retains aliases previously filtered out. Keep each nonempty `input_2`, `input_3`, … value verbatim and preserve its source-field mapping. A resource can contain multiple snippets; evidence IDs identify **resources**, not sentence offsets.

`occurrence_resource_ids` uses the existing normalized matching heuristic. `literal_resource_ids` reports exact case-sensitive occurrence. Neither is a semantic support decision. All resources are shown to the judge, so equivalent names without literal occurrence can still receive support labels.

The old 218-question review contributes partial C3 references. Question holds and matching candidate flags are imported separately. A flagged candidate only holds matching aliases, not every alias of an otherwise clear question. Original dev IDs are checked for overlap and never enter training.


In [ ]:
if "train" in globals().get("ANNOTATION_SPLITS", ["dev"]):
    packets, manifest = prep.build_packets(PROJECT_ROOT)
    assert len(packets) == 1130, "The source pool changed; inspect it before adopting this notebook's assumptions."
    assert manifest["old_alias_rows"] == 1352
    prep.initialize_run(RUN_DIR, packets, manifest)
    print(json.dumps({k: v for k, v in manifest.items() if k != "source_hashes"}, indent=2))
    print("Frozen packets:", RUN_DIR / "packets.jsonl")
else:
    print("Skipping train section (disabled in ANNOTATION_SPLITS).")


In [ ]:
if "train" in globals().get("ANNOTATION_SPLITS", ["dev"]):
    # Inspect occurrence candidates; this does not label evidence as supported.
    example = next(q for q in packets if not q["prior_question_flag"])
    print(example["question_id"], example["question"])
    for alias in example["aliases"]:
        print(alias["alias_id"], repr(alias["text"]),
              "occurs in:", alias["occurrence_resource_ids"],
              "previously flagged:", bool(alias["prior_flags"]))
    print("Resources:", len(example["resources"]))
    print("First resource:\n", example["resources"][0]["text"])
else:
    print("Skipping train section (disabled in ANNOTATION_SPLITS).")


## 3. Review the annotation prompt

An alias has two separate labels:

- **Evidence:** `supported`, `unsupported`, `conflicting`, `uncertain`.
- **Answer:** `complete`, `incomplete`, `wrong`, `uncertain`.

Export requires a clear question, supported evidence, a complete answer, exact valid citation quotes and no unresolved historical flags. Unreviewed and uncertain aliases are never relabeled unsupported. Evidence sets describe sufficient support, not an exhaustive list of positives: uncited resources are not automatically negative.

The support labels are semantic judgments by the configured annotator. Code checks IDs, coverage, status values and quote traceability; it cannot prove the judgments true.


In [ ]:
if "train" in globals().get("ANNOTATION_SPLITS", ["dev"]):
    print(prep.JUDGE_PROMPT)
else:
    print("Skipping train section (disabled in ANNOTATION_SPLITS).")


## 4. Annotate and resume

The API cell processes complete question packets one at a time. Smoke mode uses a fixed first unheld subset and at most `SMOKE_QUESTIONS` calls; no automatic retries occur. Full mode uses the larger source pool with the configured `ANNOTATION_LIMIT`. Changing the judge model, endpoint or prompt creates a separate cache signature. Old results remain intact. Provider errors stop the pass. With `CONTINUE_ON_INVALID=True`, malformed JSON, invalid quotations and incomplete responses are saved as `.error.json` and left pending while the pass continues. Each question is attempted at most once per pass. Rerunning retries pending questions and reuses valid judgments; full export remains blocked until coverage is complete. Inspect error artifacts or supply manual corrections for persistent failures. No automatic unlimited retries or hidden API calls occur.

The judge schema does not allow it to resolve prior holds. Previously held questions are skipped automatically and appear in exports as excluded. If you want to resolve one, use the manual review import below.


In [ ]:
if "train" in globals().get("ANNOTATION_SPLITS", ["dev"]):
    # Refresh the helper even when this cell is rerun in an older kernel.
    import importlib
    import inspect
    prep = importlib.reload(prep)
    if "api_key_file" not in inspect.signature(prep.chat_json_judge).parameters:
        raise RuntimeError(f"Outdated judge helper loaded from {prep.__file__}. Restart the kernel and rerun setup.")

    CACHE_DIR = None
    if ANNOTATION_BACKEND == "api":
        judge = prep.chat_json_judge(
            JUDGE_ENDPOINT, JUDGE_MODEL, API_KEY_ENV,
            max_tokens=JUDGE_MAX_OUTPUT_TOKENS, timeout=JUDGE_TIMEOUT_SECONDS,
            api_key_file=API_KEY_FILE,
        )
        # Store a hash of the endpoint, not the URL or credential.
        judge_metadata = {
            "model": JUDGE_MODEL, "endpoint_sha256": prep.digest(JUDGE_ENDPOINT),
            "max_output_tokens": JUDGE_MAX_OUTPUT_TOKENS,
            "prompt_sha256": prep.digest(prep.JUDGE_PROMPT),
        }
        annotation_packets = ([p for p in packets if not p["prior_question_flag"]][:SMOKE_QUESTIONS]
                              if SMOKE_TEST else packets)
        call_limit = SMOKE_QUESTIONS if SMOKE_TEST else ANNOTATION_LIMIT
        print("Mode:", "SMOKE" if SMOKE_TEST else "FULL", "| judge:", JUDGE_MODEL, "| maximum new calls:", call_limit)
        CACHE_DIR = prep.annotate(annotation_packets, RUN_DIR, judge, judge_metadata, limit=call_limit,
                                  continue_on_invalid=CONTINUE_ON_INVALID)
        print("Active judgment cache:", CACHE_DIR)
        completed_decisions = prep.load_decisions(packets, CACHE_DIR, MANUAL_REVIEWS)
        pending_ids = [p["question_id"] for p in packets
                       if not p["prior_question_flag"] and p["question_id"] not in completed_decisions]
        print("Unflagged questions still pending:", len(pending_ids))
        if pending_ids:
            print("Rerun this annotation cell to retry pending questions, or inspect *.error.json and add manual reviews.")
            print("Full export remains blocked until pending judgments are resolved. A partial export requires ALLOW_PARTIAL_EXPORT=True.")
    else:
        print("Manual mode: complete MANUAL_REVIEWS using the schema below; no model was called.")
else:
    print("Skipping train section (disabled in ANNOTATION_SPLITS).")


## 5. Inspect judgments and import corrections

Inspect the smoke judgments against **all** original resources before expanding the run. To finish pending questions, set `SMOKE_TEST=False` and `ANNOTATION_LIMIT=None`, then rerun from Configuration so the run directory changes as well. Do not use dev/test answers to build training labels.

Optional `manual_reviews.jsonl` contains one complete record per reviewed question. Manual records take precedence over the active judge cache, retain provenance, and must label every alias. Use the original `input_sha256` from the packet. Example envelope (fill in real values; the decision uses the exact schema printed above):

```json
{
  "question_id": "QUESTION_ID",
  "input_sha256": "COPY_FROM_PACKET",
  "origin": "human",
  "reviewer": "reviewer_identifier",
  "review_reason": "Explain checks and corrections",
  "resolve_question_flag": false,
  "resolved_alias_ids": [],
  "decision": {
    "question_id": "QUESTION_ID",
    "question_status": "clear",
    "question_reason": "Evidence establishes the requested relationship",
    "aliases": []
  }
}
```

The empty `aliases` above is a placeholder; copy/fill all alias decisions. Set `resolve_question_flag=true` only after resolving the recorded question issue. List any explicitly resolved historical alias flags in `resolved_alias_ids`. An ordinary correct-looking model answer does not clear an earlier hold.


In [ ]:
if "train" in globals().get("ANNOTATION_SPLITS", ["dev"]):
    decisions = prep.load_decisions(packets, CACHE_DIR, MANUAL_REVIEWS)
    audit = prep.audit_aliases(packets, decisions, ACCEPT_MODEL_JUDGMENTS)
    print("Annotated questions:", len(decisions), "/", len(packets))
    print("Evidence status:", dict(Counter(a["evidence_status"] for a in audit)))
    print("Answer status:", dict(Counter(a["answer_status"] for a in audit)))
    print("Eligible aliases:", sum(a["eligible_for_sft"] for a in audit))
    print("Eligible questions:", len({a["question_id"] for a in audit if a["eligible_for_sft"]}))
    # A preview is always available, even while full export is gated on coverage.
    prep.write_jsonl(RUN_DIR / "alias_decisions_preview.jsonl", audit)
    for item in list(decisions.values())[:2]:
        print(json.dumps(item["decision"], indent=2, ensure_ascii=False))
else:
    print("Skipping train section (disabled in ANNOTATION_SPLITS).")


## 6. Export matched training datasets and alias audit

Default export requires all unheld questions to be annotated. Previously flagged questions may remain explicitly excluded. Smoke mode sets `ALLOW_PARTIAL_EXPORT=True`; the summary records remaining coverage and the export is not a full-pool result.

Both arms receive identical questions, numbered full resources, resource-order permutations and answer aliases. Only the task instruction and assistant target differ. The evidence target uses the first validated sufficient set; alternative sets remain in metadata. IDs are remapped after shuffling. One permutation per question is saved; this is not automatic epoch-wise augmentation.

These are **chat-message training records**, not drop-in input for the old answer-only preprocessing function. A future training notebook must use the full assistant response as its target, and evaluation must parse the `Answer:` line separately from `Evidence:`. Use a context budget covering the complete prompt and completion; do not silently truncate resources or evidence targets.


In [ ]:
if "train" in globals().get("ANNOTATION_SPLITS", ["dev"]):
    # Reload corrections and revalidate immediately before exporting.
    ALLOW_PARTIAL_EXPORT = True
    decisions = prep.load_decisions(packets, CACHE_DIR, MANUAL_REVIEWS)
    EXPORT_DIR = RUN_DIR / "exports" / datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
    summary = prep.export_data(
        packets, decisions, EXPORT_DIR,
        accept_model_judgments=ACCEPT_MODEL_JUDGMENTS,
        all_supported_aliases=EXPORT_ALL_SUPPORTED_ALIASES,
        order_seed=RESOURCE_ORDER_SEED,
        allow_partial=ALLOW_PARTIAL_EXPORT,
    )
    print(json.dumps(summary, indent=2))
    print("Export:", EXPORT_DIR)
else:
    print("Skipping train section (disabled in ANNOTATION_SPLITS).")


In [ ]:
if "train" in globals().get("ANNOTATION_SPLITS", ["dev"]):
    evidence_rows = prep.read_jsonl(EXPORT_DIR / "evidence_sft.jsonl")
    answer_rows = prep.read_jsonl(EXPORT_DIR / "answer_only_sft.jsonl")
    assert len(evidence_rows) == len(answer_rows)
    for evidence, answer in zip(evidence_rows, answer_rows):
        assert evidence["question_id"] == answer["question_id"]
        assert evidence["messages"][1] == answer["messages"][1]
        assert evidence["metadata"]["answer"] == answer["metadata"]["answer"]
    if evidence_rows:
        print(json.dumps(evidence_rows[0], indent=2, ensure_ascii=False))
    else:
        print("No eligible training examples yet; inspect the review queue and annotation coverage.")
    for source, expected_hash in manifest["source_hashes"].items():
        assert prep.file_hash(source) == expected_hash, f"Source changed: {source}"
    print("Matched inputs/answers verified; source files unchanged.")
else:
    print("Skipping train section (disabled in ANNOTATION_SPLITS).")


## Output files

- `packets.jsonl` in the run root: all source questions, original resources, every gold alias, occurrence locations, historical hints and flags.
- `judgments/<signature>/`: resumable raw model decisions, token-usage metadata and validation errors.
- `evidence_sft.jsonl` / `answer_only_sft.jsonl`: matched training arms.
- `alias_decisions.csv` and `.jsonl`: one row per original alias, including unsupported, uncertain, incomplete, excluded and unreviewed aliases, evidence quotes and reasons.
- `question_decisions.jsonl`: per-question alias groups, exported aliases and exclusions.
- `review_queue.jsonl`: aliases not eligible for training, with reasons (including pending labels).
- `annotation_provenance.jsonl`: selected judgments and human overrides.
- `summary.json`: coverage, source of labels, export policy and output hashes.

This is an evidence-supervision pilot, not an exhaustive snippet-relevance dataset. Keep a separately reviewed dev evidence subset to measure selection quality and accept valid alternative evidence sets. Compare both arms from the same initial SFT checkpoint with matched sampling and report answer accuracy separately from evidence correctness.


## Reusable dev evidence annotations (evaluation only)

**Run All is dev-only by default.** Training cells remain guarded by `ANNOTATION_SPLITS`. This section never passes dev packets to the SFT exporter.

Annotate all **160 dev questions and 198 gold aliases**, keeping the full mixed context and the evaluator's original numbering of **2,110 snippets**. No student predictions or model identities enter annotation.

The judge now uses the **training rubric and JSON structure**: assess `evidence_status` and `answer_status`, then select one or more sufficient evidence sets with exact quotes. Dev references use numeric `snippet_id` rather than training's resource IDs. Each request handles one alias; training handles all aliases in one question together.

This is not exhaustive snippet classification. **Uncited snippets remain unreviewed, not irrelevant.** A multi-snippet set can establish an answer without each member being sufficient alone. Automatic literal/normalized occurrence is still recorded independently for all 2,610 alias–snippet pairs.

The new `full_dev_evidence_train_aligned_v2` directory separates this protocol from earlier exhaustive annotations. Old caches and exports remain intact; they are not automatically reused for the changed protocol. Rerun setup, dev settings and packet preparation before annotation. These are model judgments, not expert-validated gold; retain ambiguous cases and keep official answer MRR separate.


In [ ]:
if "dev" in globals().get("ANNOTATION_SPLITS", ["dev"]):
    from cse_dpo import dev_evidence_annotation as devprep
    prep = importlib.reload(prep)
    devprep = importlib.reload(devprep)

    DEV_RUN_NAME = "full_dev_evidence_train_aligned_v2"
    DEV_SMOKE_TEST = False
    DEV_SMOKE_QUESTIONS = 1  # Fixed first question(s), including every alias of each.
    DEV_RUN_DIR = PROJECT_ROOT / "Artifacts/Factoid_SFT/dev_evidence_annotation" / (DEV_RUN_NAME + ("_smoke" if DEV_SMOKE_TEST else ""))
    DEV_ANNOTATION_BACKEND = "api"  # "api" or "manual"
    DEV_JUDGE_ENDPOINT = os.environ.get("EVIDENCE_JUDGE_ENDPOINT", "https://api.openai.com/v1/chat/completions")
    DEV_JUDGE_MODEL = os.environ.get("EVIDENCE_JUDGE_MODEL", "gpt-4.1-mini-2025-04-14")
    DEV_API_KEY_FILE = PROJECT_ROOT / "open_ai_api.txt"  # Never printed or stored.
    DEV_API_KEY_ENV = "EVIDENCE_JUDGE_API_KEY"
    DEV_MAX_OUTPUT_TOKENS = 6000  # Same output budget as train: selected sufficient evidence sets.
    DEV_TIMEOUT_SECONDS = 180
    DEV_ANNOTATION_LIMIT = None  # Maximum NEW per-alias calls this execution; None = all pending.
    DEV_CONTINUE_ON_INVALID = True
    DEV_VALIDATION_RETRIES = 2  # Initial request + two correction retries (maximum 3 calls per alias).
    DEV_FALLBACK_TO_EXACT_MATCH = True  # Save literal occurrence labels after all 3 validation failures.
    DEV_MANUAL_REVIEWS = DEV_RUN_DIR / "manual_reviews.jsonl"
    DEV_ALLOW_PARTIAL_EXPORT = False  # True for a deliberately incomplete/smoke snapshot.
    assert DEV_RUN_NAME and DEV_RUN_NAME not in {".", ".."} and not any(c in DEV_RUN_NAME for c in "/\\")
    assert DEV_ANNOTATION_BACKEND in {"api", "manual"}
    print("Dev-only output:", DEV_RUN_DIR)
    print("Mode:", "SMOKE" if DEV_SMOKE_TEST else "FULL")
else:
    print("Skipping dev section (disabled in ANNOTATION_SPLITS).")


In [ ]:
if "dev" in globals().get("ANNOTATION_SPLITS", ["dev"]):
    dev_packets, dev_manifest = devprep.build_packets(PROJECT_ROOT)
    assert dev_manifest["question_count"] == 160
    assert dev_manifest["train_dev_disjoint"]
    devprep.initialize_run(DEV_RUN_DIR, dev_packets, dev_manifest)
    dev_tasks = devprep.tasks_for(dev_packets)
    prep.write_jsonl(DEV_RUN_DIR / "annotation_tasks.jsonl", dev_tasks)
    print(json.dumps({k:v for k,v in dev_manifest.items() if k != "source_hashes"}, indent=2))
    print("Automatic occurrence labels:", DEV_RUN_DIR / "alias_snippet_occurrences.jsonl")
    print("Semantic requests: one per question-alias;", len(dev_tasks), "total before cache reuse.")
    print(devprep.JUDGE_PROMPT)
else:
    print("Skipping dev section (disabled in ANNOTATION_SPLITS).")


### Annotate sufficient evidence sets and resume

The next cell sends the question, one proposed alias, occurrence hints and full mixed context to the configured judge, using the existing key file. The explicit `snippets[]` table maps IDs to exact text. Read all snippets, but return only sufficient sets with short, exact quotes. The prompt is derived from the training helper; it no longer requests a label and explanation for every snippet.

Valid tasks in this new protocol's cache are reused on rerun. Changing the judge or prompt creates a separate cache. Invalid responses receive **two correction retries** after the initial call (three calls maximum per alias). Recorded failures count toward this cap across reruns. Quotes and IDs are never silently repaired. Provider/network failures stop the run while preserving completed annotations.

After three invalid responses, `DEV_FALLBACK_TO_EXACT_MATCH=True` saves case-sensitive, token-bounded gold-alias occurrences inside snippet bodies. An empty match list is allowed. Fallbacks remain **semantically unreviewed**, are cached, and may be replaced by a human correction. `DEV_ANNOTATION_LIMIT` counts retries too; hitting it before three failures leaves an alias pending. A smoke run always covers the same first question(s), including their aliases.


In [ ]:
if "dev" in globals().get("ANNOTATION_SPLITS", ["dev"]):
    devprep = importlib.reload(devprep)
    DEV_CACHE_DIR = None
    if DEV_ANNOTATION_BACKEND == "api":
        dev_judge = prep.chat_json_judge(
            DEV_JUDGE_ENDPOINT, DEV_JUDGE_MODEL, DEV_API_KEY_ENV,
            max_tokens=DEV_MAX_OUTPUT_TOKENS, timeout=DEV_TIMEOUT_SECONDS,
            api_key_file=DEV_API_KEY_FILE, system_prompt=devprep.JUDGE_PROMPT,
        )
        dev_judge_metadata = {"model":DEV_JUDGE_MODEL,
                              "endpoint_sha256":prep.digest(DEV_JUDGE_ENDPOINT),
                              "max_output_tokens":DEV_MAX_OUTPUT_TOKENS}
        DEV_CACHE_DIR = devprep.annotate(
            dev_packets, DEV_RUN_DIR, dev_judge, dev_judge_metadata,
            limit=DEV_ANNOTATION_LIMIT,
            smoke_questions=DEV_SMOKE_QUESTIONS if DEV_SMOKE_TEST else None,
            continue_on_invalid=DEV_CONTINUE_ON_INVALID,
            validation_retries=DEV_VALIDATION_RETRIES,
            fallback_to_exact_match=DEV_FALLBACK_TO_EXACT_MATCH,
        )
    else:
        print("Manual mode: fill DEV_MANUAL_REVIEWS; no model was called.")
    dev_decisions = devprep.load_decisions(dev_packets, DEV_CACHE_DIR, DEV_MANUAL_REVIEWS)
    print("Completed aliases (semantic or fallback):", len(dev_decisions), "/", len(dev_tasks))
    print("Exact-match fallbacks:", sum(v["origin"] == "exact_match_fallback" for v in dev_decisions.values()))
    print("Pending aliases:", len(dev_tasks) - len(dev_decisions))
    print("Judgment cache:", DEV_CACHE_DIR)
else:
    print("Skipping dev section (disabled in ANNOTATION_SPLITS).")


### Review and freeze an evaluation snapshot

Manual corrections in `DEV_MANUAL_REVIEWS` override the active cache. One JSONL record per task:

```json
{
  "task_id": "QUESTION_ID__a1",
  "input_sha256": "COPY_FROM_annotation_tasks.jsonl",
  "origin": "human",
  "reviewer": "reviewer_identifier",
  "review_reason": "What was checked or corrected",
  "decision": {
    "question_id": "QUESTION_ID",
    "question_status": "clear",
    "question_reason": "The requested fact is unambiguous.",
    "aliases": [{
      "alias_id": "a1", "evidence_status": "supported", "answer_status": "complete",
      "rationale": "Why this alias answers the question.",
      "evidence_sets": [[{"snippet_id": 1, "quote": "REPLACE_WITH_EXACT_SOURCE_QUOTE", "reason": "How the quote supports the fact."}]]
    }]
  }
}
```

Replace the example IDs, text and judgments with actual reviewed values. Review scope, qualifiers, alternative interpretations and conflicting evidence. Each export is a new timestamped snapshot.

Full export requires every alias to have a semantic judgment or cached fallback. `semantic_alias_annotations_complete` describes alias-task coverage; `semantic_labels_complete` additionally requires every alias–snippet pair to be reviewed, so it will normally be **false** with selected-set annotations. That is expected. Unselected snippets remain unreviewed even when all alias tasks are finished.

Fallbacks have `origin="exact_match_fallback"`; their `fallback_match_snippet_ids`, `fallback_evidence_sets`, and per-pair `fallback_occurrence_label` are weak labels. They do not populate semantic sufficient sets. Report fallback coverage separately. Use `DEV_ALLOW_PARTIAL_EXPORT=True` only for an intentional partial/smoke snapshot.


In [ ]:
if "dev" in globals().get("ANNOTATION_SPLITS", ["dev"]):
    dev_decisions = devprep.load_decisions(dev_packets, DEV_CACHE_DIR, DEV_MANUAL_REVIEWS)
    DEV_EXPORT_DIR = DEV_RUN_DIR / "exports" / datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
    dev_summary = devprep.export_annotations(
        dev_packets, dev_manifest, dev_decisions, DEV_EXPORT_DIR,
        allow_partial=DEV_ALLOW_PARTIAL_EXPORT,
    )
    print(json.dumps({k:v for k,v in dev_summary.items() if k not in {"source_manifest", "output_hashes"}}, indent=2))
    print("Reusable dev reference:", DEV_EXPORT_DIR)
    assert not (DEV_EXPORT_DIR / "evidence_sft.jsonl").exists()
    assert not (DEV_EXPORT_DIR / "answer_only_sft.jsonl").exists()
else:
    print("Skipping dev section (disabled in ANNOTATION_SPLITS).")


### Dev output files

- `dev_packets.jsonl`: frozen questions, full resources, snippet numbering and original gold aliases.
- `dev_snippet_index.jsonl`: exact snippet text, source resource/field, PubMed ID and offsets.
- `dev_snippet_alias_labels.jsonl`: all alias–snippet pairs with occurrence flags, semantic status, quote and rationale.
- `dev_alias_labels.jsonl`: answer status, supporting snippet IDs, sufficient singleton/joint evidence sets and uncertainty.
- `dev_question_labels.jsonl`: annotation coverage and question ambiguity across aliases.
- `dev_annotation_provenance.jsonl`: model decisions and any human overrides.
- `dev_review_queue.jsonl`: pending, ambiguous, uncertain or incomplete annotations.
- `summary.json`: split, coverage, source fingerprints and output hashes.

Use a frozen snapshot for automatic comparison of **known sufficient sets**, conditional on the answer alias. An unlisted snippet or set is **unknown**, not automatically incorrect. These sparse labels cannot establish exhaustive citation precision or recall; additional review is needed for unfamiliar citations or novel answer variants. Keep occurrence agreement separate from semantic evidence correctness. No per-answer judge is needed for reference-covered cases; the optional per-output judge can investigate uncovered cases.
